# Market Narrative Pipeline — Colab quickstart
Runtime → GPU (T4 is enough). Colab secrets: `FRED_API_KEY`, `FMP_API_KEY`, optional `BENZINGA_API_KEY`, `HF_TOKEN`, plus whatever your own data connector needs.

In [36]:
import os, sys, subprocess
BRANCH = 'claude/quirky-wozniak-72igbq'
REPO = '/content/OpenBB'
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode: print('FAILED:', cmd, '\n', r.stderr[-500:])
    return r.stdout.strip()
if not os.path.isdir(REPO + '/.git'):
    sh(f'git clone -q --depth 1 --single-branch -b {BRANCH} https://github.com/jam2205/OpenBB.git {REPO}')
else:
    # Force the checkout to match GitHub exactly (discards local edits inside the clone).
    sh(f'git -C {REPO} fetch -q --depth 1 origin {BRANCH}')
    sh(f'git -C {REPO} reset -q --hard FETCH_HEAD')
print('Code version on disk:', sh(f'git -C {REPO} log -1 --format="%h %s"'))
%cd /content/OpenBB/pipelines/market_narrative
# Drop any already-imported copy so the next import uses the code on disk (no restart needed).
stale = [m for m in sys.modules if m == 'mnp' or m.startswith('mnp.')]
for m in stale: del sys.modules[m]
print(f'Cleared {len(stale)} cached pipeline modules')
!pip -q install -r requirements.txt 2>&1 | grep -v "dependency resolver\|incompatible" | tail -2

Code version on disk: adb2dcb Pair briefs: drift-keeping control, and top-up for pairs whose candles lag
/content/OpenBB/pipelines/market_narrative
Cleared 16 cached pipeline modules


In [17]:
import os
from google.colab import userdata
# Canonical name -> names it may be saved under in Colab Secrets (first found wins). Add yours if different.
ALIASES = {
    "FRED_API_KEY": ["FRED_API_KEY", "FRED_KEY", "API_FRED_KEY", "fred_api_key"],
    "FMP_API_KEY": ["FMP_API_KEY", "FMP_KEY", "fmp_api_key"],
    "BENZINGA_API_KEY": ["BENZINGA_API_KEY", "BENZINGA_KEY"],
    "HF_TOKEN": ["HF_TOKEN", "HUGGINGFACE_TOKEN", "HF_API_KEY"],
}
for canon, names in ALIASES.items():
    for n in names:
        try:
            v = userdata.get(n)
        except Exception:
            continue
        if v:
            os.environ[canon] = v
            break
from openbb import obb
for env, attr in [("FRED_API_KEY", "fred_api_key"), ("FMP_API_KEY", "fmp_api_key"), ("BENZINGA_API_KEY", "benzinga_api_key")]:
    if os.environ.get(env):
        setattr(obb.user.credentials, attr, os.environ[env])
print({k: bool(os.environ.get(k)) for k in ALIASES})

{'FRED_API_KEY': True, 'FMP_API_KEY': True, 'BENZINGA_API_KEY': True, 'HF_TOKEN': True}


## 0. Connect your own data store (optional but recommended)
Paste your private connector cell below and run it (the one that ends with `5/5 connected: ... tables`). It must define `read(name, ...)` and `tables()`. It is not stored here because this repository is public.

With it, the report uses **your own economic calendar** (all central banks, auctions, forecasts with release times), **global 10Y yields**, **FX carry** and **COT positioning**. Without it, the calendar falls back to Benzinga, then FRED.

In [22]:
"""First cell of any Colab notebook that reads the Jetsons buckets (version 2, 2026-10-13).

Paste this whole file into the first cell and run it. It is the same for every
notebook. It prints a line for every step, so a run that goes wrong says where.

Secrets it needs (Colab's key icon, "Notebook access" on):
  TS_AUTHKEY         Tailscale auth key (Reusable + Ephemeral)
  MINIO_ACCESS_KEY   colab-reader
  MINIO_SECRET_KEY   the colab-reader secret

What you get:
  tables()                         names of the Jetson's current tables
  read("candles_1h")               a table as a DataFrame
  read("<pull>", bucket="tv-pulls")  a chart pull
  save(df, "my_test/result")       write a result to colab-scratch

How it reads: each file is DOWNLOADED once to /content/jetson_cache and read
from there. The link to the Jetson is relayed and slow (a 90 MB table takes
minutes), and a plain download with retries is what survives that; the first
version read the remote file piecemeal and stalled.

Version 1 used s3fs over the proxy. Changed after the first real Colab run:
the notebook joined the private network but no data ever crossed it.
"""
import os, subprocess, time, json

JETSON_NAME = "jetson-fx.tail7f5b29.ts.net"   # the Jetson's fixed name on the private network
JETSON_IP = "100.112.179.107"                 # the same machine by address, used if the name does not resolve
PROXY = "localhost:1055"                      # Colab cannot open a network device; traffic goes through this local proxy
TS_DIR = "/content/tailscale"
CACHE = "/content/jetson_cache"
TS = f"{TS_DIR}/tailscale --socket={TS_DIR}/tailscaled.sock"

from google.colab import userdata

def _sh(cmd, timeout=120):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)

# -- 0. secrets ------------------------------------------------------------------
_missing = []
for _n in ("TS_AUTHKEY", "MINIO_ACCESS_KEY", "MINIO_SECRET_KEY"):
    try:
        if not userdata.get(_n):
            _missing.append(_n)
    except Exception:
        _missing.append(_n)
assert not _missing, f"Secrets missing or without notebook access: {_missing}. Open the key icon on the left and check names and the access switch."
print("1/5 secrets found")

# -- 1. Tailscale ------------------------------------------------------------------
if not os.path.exists(f"{TS_DIR}/tailscaled"):
    os.makedirs(TS_DIR, exist_ok=True)
    _name = _sh("curl -fsSL 'https://pkgs.tailscale.com/stable/?mode=json' | python3 -c "
                "\"import json,sys; print(json.load(sys.stdin)['Tarballs']['amd64'])\"").stdout.strip()
    _got = _sh(f"curl -fsSL https://pkgs.tailscale.com/stable/{_name} | tar xz -C {TS_DIR} --strip-components=1")
    assert _got.returncode == 0 and os.path.exists(f"{TS_DIR}/tailscaled"), f"could not download Tailscale: {_got.stderr[:300]}"
print("2/5 Tailscale downloaded")

if _sh("pgrep -f '[t]ailscaled --tun=userspace-networking'").returncode != 0:
    _sh(f"rm -f {TS_DIR}/tailscaled.sock")
    subprocess.Popen(
        f"nohup {TS_DIR}/tailscaled --tun=userspace-networking --state=mem: --socket={TS_DIR}/tailscaled.sock "
        f"--socks5-server={PROXY} --outbound-http-proxy-listen={PROXY} > /content/tailscaled.log 2>&1 &",
        shell=True, start_new_session=True)
    for _ in range(40):
        if os.path.exists(f"{TS_DIR}/tailscaled.sock"):
            break
        time.sleep(0.5)
assert os.path.exists(f"{TS_DIR}/tailscaled.sock"), "tailscaled did not start: " + _sh("tail -5 /content/tailscaled.log").stdout

def _state():
    try:
        return json.loads(_sh(f"{TS} status --json").stdout).get("BackendState")
    except Exception:
        return None

if _state() != "Running":
    _up = _sh(f"{TS} up --authkey={userdata.get('TS_AUTHKEY')} --hostname=colab-notebook --timeout=60s")
    assert _up.returncode == 0, ("could not join the private network: " + (_up.stderr or _up.stdout)[:400]
                                 + "\nThe TS_AUTHKEY may be expired or not marked Reusable.")
assert _state() == "Running", "Tailscale is not running: " + _sh(f"{TS} status").stdout[:300]
print("3/5 joined the private network")

# -- 2. can we actually reach the Jetson through the proxy? ------------------------
def _alive(host):
    r = _sh(f"curl -s -o /dev/null -m 25 -w '%{{http_code}}' -x http://{PROXY} http://{host}:9000/minio/health/live", timeout=40)
    return r.stdout.strip() == "200"

JETSON = None
for _try in range(4):                      # the first packets to a new peer can take a few seconds
    for _host in (JETSON_NAME, JETSON_IP):
        if _alive(_host):
            JETSON = _host
            break
    if JETSON:
        break
    time.sleep(5)
assert JETSON, ("joined the network but the Jetson's MinIO does not answer through the proxy.\n"
                + _sh(f"{TS} status").stdout[:400] + "\n" + _sh(f"{TS} ping -c 2 jetson-fx", timeout=40).stdout[:300])
print(f"4/5 the Jetson answers at {JETSON}:9000")

# -- 3. the buckets ----------------------------------------------------------------
try:
    import boto3
except ImportError:
    _sh("pip -q install boto3", timeout=300)
    import boto3
import pandas as pd
from botocore.config import Config

_s3 = boto3.client(
    "s3", endpoint_url=f"http://{JETSON}:9000", region_name="us-east-1",
    aws_access_key_id=userdata.get("MINIO_ACCESS_KEY"), aws_secret_access_key=userdata.get("MINIO_SECRET_KEY"),
    config=Config(proxies={"http": f"http://{PROXY}"}, s3={"addressing_style": "path"},
                  retries={"max_attempts": 8, "mode": "standard"}, connect_timeout=30, read_timeout=300),
)

def _key(name, bucket):
    return f"current/{name}.parquet" if bucket == "medallion-serving" else f"{name}.parquet"

def tables(bucket="medallion-serving"):
    """Names you can pass to read()."""
    prefix = "current/" if bucket == "medallion-serving" else ""
    out = []
    for page in _s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
        out += [o["Key"][len(prefix):-8] for o in page.get("Contents", []) if o["Key"].endswith(".parquet")]
    return sorted(out)

def fetch(name, bucket="medallion-serving", fresh=False):
    """Download one file to /content/jetson_cache (once per session unless fresh=True) and return its path."""
    key = _key(name, bucket)
    local = os.path.join(CACHE, bucket, key)
    size = _s3.head_object(Bucket=bucket, Key=key)["ContentLength"]
    if fresh or not os.path.exists(local) or os.path.getsize(local) != size:
        os.makedirs(os.path.dirname(local), exist_ok=True)
        t0 = time.time()
        print(f"downloading {bucket}/{key} ({size / 1e6:.1f} MB) ...", end=" ", flush=True)
        _s3.download_file(bucket, key, local + ".part")
        os.replace(local + ".part", local)
        print(f"done in {time.time() - t0:.0f}s")
    return local

def read(name, bucket="medallion-serving", columns=None, filters=None, fresh=False):
    """A table as a DataFrame: read('candles_1h'), or read('<pull name>', bucket='tv-pulls')."""
    return pd.read_parquet(fetch(name, bucket, fresh), columns=columns, filters=filters)

def save(df, name):
    """Write a result to colab-scratch, the one bucket this login may write to."""
    local = os.path.join(CACHE, "_out.parquet")
    os.makedirs(CACHE, exist_ok=True)
    df.to_parquet(local, index=False)
    _s3.upload_file(local, "colab-scratch", f"{name}.parquet")
    return f"colab-scratch/{name}.parquet"

_t, _p = tables(), tables("tv-pulls")
try:
    _stamp = json.loads(_s3.get_object(Bucket="medallion-serving", Key="current/MANIFEST.json")["Body"].read()).get("generated_at", "?")[:16]
except Exception:
    _stamp = "?"
print(f"5/5 connected: {len(_t)} tables in medallion-serving (snapshot {_stamp} UTC), {len(_p)} chart pull(s) in tv-pulls")
print("Use: tables()   read('range_forecast_24h')   read(tables('tv-pulls')[0], bucket='tv-pulls')   save(df, 'name')")

1/5 secrets found
2/5 Tailscale downloaded
3/5 joined the private network
4/5 the Jetson answers at jetson-fx.tail7f5b29.ts.net:9000
5/5 connected: 65 tables in medallion-serving (snapshot 2026-11-10T22:32 UTC), 1 chart pull(s) in tv-pulls
Use: tables()   read('range_forecast_24h')   read(tables('tv-pulls')[0], bucket='tv-pulls')   save(df, 'name')


In [23]:
from mnp import medallion
# Each role is matched to one of your datasets by name. To choose yourself: OVERRIDE = {'calendar': '<dataset>'}.
# To switch a role off: {'positioning': ''}. Nothing is downloaded until a report section needs it.
OVERRIDE = {}
try:
    roles = medallion.register(read, tables, datasets=OVERRIDE)
    print('reader registered:', len(tables()), 'datasets')
    for r in ('calendar', 'yields', 'carry', 'positioning', 'candles'):
        print(f'  {r:12s} -> {roles.get(r, "(none found)")}')
except NameError:
    print('Connector cell not run -> public calendars only (Benzinga, then FRED)')

reader registered: 65 datasets
  calendar     -> calendar_events
  yields       -> world_bond_yields
  carry        -> carry_features
  positioning  -> cot_pair_sentiment
  candles      -> candles_1h


### What the pipeline will read from your store
This week's calendar, global yields and positioning exactly as the report will see them. An empty table means that role's dataset is missing a column the pipeline expects (listed at the top of `mnp/medallion.py`).

In [24]:
from datetime import date, timedelta
from mnp import medallion
from mnp.run import load_config
cfg = load_config(); c = cfg['calendar']; fx = cfg['universe']['fx_openbb'] + cfg['universe']['fx_minio']
display(medallion.calendar(date.today(), date.today() + timedelta(days=7), c['currencies'], tuple(c['impacts'])))
display(medallion.global_yields(date.today()))
display(medallion.positioning(fx))
display(medallion.carry(fx))

downloading medallion-serving/current/calendar_events.parquet (0.1 MB) ... done in 1s


,date,currency,event,impact,event_code,family,forecast,previous,actual
0,2026-11-10 13:30:00+00:00,USD,CPI m/m,high,CPI,inflation,0.3%,0.6%,
1,2026-11-10 13:30:00+00:00,USD,Core CPI m/m,high,CORE_CPI,inflation,0.2%,0.3%,
2,2026-11-12 07:00:00+00:00,GBP,GDP m/m,high,GDP,growth,0.0%,0.1%,
3,2026-11-13 13:30:00+00:00,USD,Core PPI m/m,high,PPI,inflation,0.3%,0.7%,


downloading medallion-serving/current/world_bond_yields.parquet (1.1 MB) ... done in 6s


,10y_%,chg_1w_bp,chg_1m_bp,vs_US_bp,2s10s_bp,as_of
US,4.88,-3.0,-36.0,0.0,32.0,2026-11-09
AU,5.08,-13.0,-25.0,20.0,26.0,2026-11-04
JP,2.72,-9.0,-30.0,-216.0,95.0,2026-11-09


downloading medallion-serving/current/cot_pair_sentiment.parquet (0.4 MB) ... done in 2s


,institutional_bias,bias_direction,base_index,quote_index,report_age_days,report_date
pair,,,,,,
EURUSD,1.02,LONG_BASE,1.02,NaN,7,2026-11-03
USDJPY,-0.09,NEUTRAL,NaN,0.09,7,2026-11-03


downloading medallion-serving/current/carry_features.parquet (0.5 MB) ... done in 1s


,carry,carry_zscore_20d,carry_zscore_90d,carry_momentum_20d,carry_momentum_5d,date
pair,,,,,,
EURUSD,-1.375,NaN,-0.106,0.0,0.0,2026-11-10 00:00:00+00:00
USDJPY,2.625,NaN,-0.291,0.0,0.0,2026-11-10 00:00:00+00:00


## 1. AcyLa predictive model: skill table + live signals
LightGBM-only by default. Signals with test dir. accuracy < 55% show as "no skill".

In [25]:
from datetime import date, timedelta
from mnp.predictive import AcyLaModel
from mnp import sources
m = AcyLaModel('AcyLa/multi-asset-predictive-model')
start = date.today() - timedelta(days=900)
m.predict_many({t: sources.ohlcv(t, start, 'yfinance') for t in ['SPY', 'QQQ', 'GLD']})

,ticker,horizon_d,pred_return,model_signal,test_dir_acc,sharpe_adj,shown_signal
0,SPY,1,0.000237,HOLD,0.567870,0.948922,HOLD
1,SPY,5,0.001599,HOLD,0.625156,1.501338,HOLD
2,SPY,21,0.006798,HOLD,0.719801,1.576815,HOLD
3,QQQ,1,-0.001001,HOLD,0.542964,1.368236,no skill
4,QQQ,5,0.002238,HOLD,0.594022,1.311810,HOLD
5,QQQ,21,0.010564,BUY,0.687422,1.362992,BUY
6,GLD,1,0.000489,HOLD,0.477650,-0.194911,no skill
7,GLD,5,-0.001242,HOLD,0.406130,-1.112077,no skill
8,GLD,21,0.003168,HOLD,0.403576,-0.891803,no skill


## 2. Smoke-test the pre-trained NLP models (no training needed)

In [26]:
from mnp.nlp import NLP
from mnp.run import load_config
import pandas as pd
cfg = load_config(); nlp = NLP(cfg['models'])
news = pd.DataFrame({'text': ['Gold rallies as yields fall after soft CPI', 'Nvidia guidance disappoints',
                              'BoJ hints at yen intervention', 'Treasury auction tails, 10Y jumps 8bp']})
nlp.attribute_drivers(news, cfg['drivers'])[['text','tone','signed','driver','driver_sim','driver_method']]

,text,tone,signed,driver,driver_sim,driver_method
0,Gold rallies as yields fall after soft CPI,positive,0.910737,metals,0.632031,zero-shot
1,Nvidia guidance disappoints,negative,-0.999993,earnings,0.996867,zero-shot
2,BoJ hints at yen intervention,neutral,0.000000,fx,0.787635,zero-shot
3,"Treasury auction tails, 10Y jumps 8bp",neutral,0.000000,rates,0.782217,zero-shot


## 3. Full run → regime-specific report

In [27]:
import logging; logging.basicConfig(level=logging.WARNING)
from mnp.run import run
from mnp import report
path = run(out_dir='reports')
report.show(path)   # shown from the HTML file: a notebook's Markdown view turns "$210,000 ... $7,943" into a formula

## 3b. Per-pair session briefs (needs your data store)
For each pair in `config.yaml → pairs.list`: how much of today's expected range is used, how often the day's low and high held from a state like this one (beside the plain rate **and a random walk**), the week's extremes, and the calendar, yield gap, carry, positioning and regime label for its two currencies. The first run downloads your hourly candles once (large file, a few minutes over the relay).

In [28]:
from mnp import pairs, report
res = pairs.run(out_dir='reports/pairs')
cols = ['pair', 'error', 'template', 'stale', 'used_of_expected', 'low_verdict', 'p_low_in', 'p_low_in_random_walk', 'low_vs_random_walk',
        'high_verdict', 'p_high_in', 'p_high_in_random_walk', 'high_vs_random_walk', 'events_before_roll']
display(res['table'][[c for c in cols if c in res['table']]])
if res['briefs']:
    display(report.show(next(iter(res['briefs'].values()))))   # the first pair's full brief; the rest are in reports/pairs/
else:
    print('No brief was written: the "error" column above says why for each pair.')

downloading medallion-serving/current/candles_1h.parquet (96.1 MB) ... done in 32s
downloading medallion-serving/current/range_forecast_24h.parquet (0.7 MB) ... done in 1s
downloading medallion-serving/current/markov_regime_log.parquet (0.1 MB) ... done in 1s


,pair,template,stale,used_of_expected,low_verdict,p_low_in,p_low_in_random_walk,low_vs_random_walk,high_verdict,p_high_in,p_high_in_random_walk,high_vs_random_walk,events_before_roll
0,EURGBP,range_spent,False,1.298315,likely in,0.993837,0.993343,same as a random walk,not established,0.920833,0.928294,same as a random walk,0
1,EURAUD,range_open,False,0.823748,likely in,0.980132,0.979624,same as a random walk,not established,0.945148,0.911049,above a random walk,0
2,EURNZD,range_open,False,0.623773,not established,0.938202,0.944276,same as a random walk,not established,0.887850,0.879074,same as a random walk,0
3,EURUSD,range_spent,False,1.447201,not established,0.959677,0.947053,same as a random walk,not established,0.998415,0.994717,above a random walk,0
4,EURCAD,range_open,False,0.771921,likely in,0.983302,0.971425,same as a random walk,likely in,0.985102,0.972026,above a random walk,0
5,EURCHF,range_open,False,0.877249,not established,0.976231,0.976885,same as a random walk,not established,0.933071,0.910375,same as a random walk,0
6,EURJPY,range_open,False,0.708052,likely in,0.985586,0.974515,above a random walk,likely in,0.969407,0.976203,same as a random walk,0
7,GBPAUD,range_open,False,0.835430,likely in,0.981102,0.976874,same as a random walk,likely in,0.985849,0.980104,same as a random walk,0
8,GBPNZD,range_open,False,0.551006,not established,0.925926,0.938292,same as a random walk,not established,0.934247,0.939839,same as a random walk,0
9,GBPUSD,range_spent,False,1.707931,not established,0.967742,0.953189,same as a random walk,not established,0.998379,0.995711,same as a random walk,0


## IMF (no key): find indicator symbols, then paste them into `config.yaml → imf.indicators`

In [29]:
from mnp import sources
res = sources.imf_search('exchange rate')
res.head(30)  # copy the `symbol` values you want (format DATAFLOW::INDICATOR)

,symbol_root,symbol,description,agency_id,dataflow_id,dataflow_name,structure_id,dimension_id,long_description,member_of
0,T111ESN,PCPS::T111ESN,"US dollar, SDR per US dollar, Period average",IMF.RES,PCPS,Primary Commodity Price System (PCPS),DSD_PCPS,INDICATOR,Exchange rate indicating how many Special Draw...,[]
1,T112EREP,PCPS::T112EREP,"Pound sterling, Domestic currency per US dolla...",IMF.RES,PCPS,Primary Commodity Price System (PCPS),DSD_PCPS,INDICATOR,Domestic Currency per US Dollar refers to the ...,[]
2,T132EREP,PCPS::T132EREP,"French franc, Domestic currency per US dollar,...",IMF.RES,PCPS,Primary Commodity Price System (PCPS),DSD_PCPS,INDICATOR,Domestic Currency per US Dollar refers to the ...,[]
3,T134EREP,PCPS::T134EREP,"Deutsche mark, Domestic currency per US dollar...",IMF.RES,PCPS,Primary Commodity Price System (PCPS),DSD_PCPS,INDICATOR,Domestic Currency per US Dollar refers to the ...,[]
4,T142EREP,PCPS::T142EREP,"Norwegian krone, Domestic currency per US doll...",IMF.RES,PCPS,Primary Commodity Price System (PCPS),DSD_PCPS,INDICATOR,Domestic Currency per US Dollar refers to the ...,[]
5,T144EREP,PCPS::T144EREP,"Swedish krona, Domestic currency per US dollar...",IMF.RES,PCPS,Primary Commodity Price System (PCPS),DSD_PCPS,INDICATOR,Domestic Currency per US Dollar refers to the ...,[]
6,T156EREP,PCPS::T156EREP,"Canadian dollar, Domestic currency per US doll...",IMF.RES,PCPS,Primary Commodity Price System (PCPS),DSD_PCPS,INDICATOR,Domestic Currency per US Dollar refers to the ...,[]
7,T158EREP,PCPS::T158EREP,"Japanese yen, Domestic currency per US dollar,...",IMF.RES,PCPS,Primary Commodity Price System (PCPS),DSD_PCPS,INDICATOR,Domestic Currency per US Dollar refers to the ...,[]
8,T163EREP,PCPS::T163EREP,"Euro, Domestic currency per US dollar, Period ...",IMF.RES,PCPS,Primary Commodity Price System (PCPS),DSD_PCPS,INDICATOR,Domestic Currency per US Dollar refers to the ...,[]
9,T193EREP,PCPS::T193EREP,"Australian dollar, Domestic currency per US do...",IMF.RES,PCPS,Primary Commodity Price System (PCPS),DSD_PCPS,INDICATOR,Domestic Currency per US Dollar refers to the ...,[]


## Swiss National Bank (no key)
The report's Switzerland / CHF section, on its own. `snb.dimensions('<cube>')` lists the series in any other cube on data.snb.ch.

In [30]:
from datetime import date
from mnp import snb
b = snb.block(date.today())
print('as of:', b['asof'], '| notes:', b['notes'])
for k in ['policy', 'sight', 'fx_tx', 'eff_fx', 'curve', 'global_rates', 'reserves']:
    print('\n', k); display(b[k])

as of: {'policy': datetime.date(2026, 11, 6), 'sight': datetime.date(2026, 11, 6), 'curve': datetime.date(2026, 10, 30), 'eff_fx': datetime.date(2026, 10, 30), 'reserves': '2026Q3', 'global_rates': '2026-09'} | notes: []

 policy


,last_%,chg_1m_pp,chg_3m_pp
SNB policy rate,0.00,0.0,0.00
SARON,-0.04,0.0,0.01
Sight deposits above threshold,-0.25,0.0,0.00
Sight deposits up to threshold,0.00,0.0,0.00



 sight


,last_chf_bn,chg_1w,chg_4w,chg_13w
Total sight deposits,452.89,-7.27,-3.50,-9.50
Domestic banks,429.15,-7.74,-3.44,-3.49
Other sight deposits,23.74,0.47,-0.06,-6.01



 fx_tx


,fx_purchases_chf_bn
quarter,
2025Q1,0.05
2025Q2,5.06
2025Q3,0.08
2025Q4,-0.01
2026Q1,3.94
2026Q2,1.44


... (4 outputs left out)


 global_rates


,rate_%,chg_1y_pp,vs_SNB_pp
SNB policy rate,0.00,0.00,0.00
Fed target (mid),3.88,-0.25,3.88
ECB deposit facility,2.50,0.50,2.50
BoE Bank rate,3.75,-0.25,3.75
BoJ call rate,1.25,0.75,1.25



 reserves


,share_%,chg_1y_pp
USD,35.8,-0.1
EUR,39.9,-0.3
JPY,8.2,0.7
GBP,5.8,-0.5
CAD,2.7,0.1
Other,7.6,0.2


## FMP: which endpoints does your plan serve?
`probe()` checks 46 core endpoints, `probe(extended=True)` about 85 more. Reports `ok` / `plan` (not on your plan) / `missing` / `empty`. Your key is never printed. Re-run if your plan changes.

In [31]:
import pandas as pd
from mnp.fmp_probe import probe
pd.set_option('display.max_rows', 200, 'display.max_colwidth', 80, 'display.width', 250)
print(probe().to_string(index=False))
# print(probe(extended=True).to_string(index=False))

FMP probe: 29 ok, 14 plan, 3 empty
         use                               endpoint                            path status  rows                                                                                                                                      fields / note
    calendar                      Earnings calendar               earnings-calendar     ok    11                                                                symbol, date, epsActual, epsEstimated, revenueActual, revenueEstimated, lastUpdated
    calendar                      Economic calendar               economic-calendar   plan     0                                                                                                                                                   
    calendar                     Dividends calendar              dividends-calendar     ok     9                                                                      symbol, date, recordDate, paymentDate, declarationDate, adjDividend

## Benzinga: check the key before training
About 60 requests. Shows which endpoints the key serves, how far back news goes, whether articles have bodies, which channels and tags they carry (candidate training labels), coverage of FX / commodities / rates, and whether the economic calendar has `actual` and `consensus`. **Run this before building a training set** and paste the printout back.

In [32]:
from mnp.benzinga_probe import probe
bz = probe()


===== 1. ENDPOINTS =====
         use                   endpoint                                      path           status  rows                                                                                                                          fields
0       news              News (latest)                              /api/v2/news               ok     5                                                                          id, author, created, updated, title, teaser, body, url
1       news             News by ticker                              /api/v2/news            empty     0                                                                                                                                
2       news         News channels list                     /api/v2/news/channels  error: http 400     0                                                                                                                                
3       news  News removed/updated feed   

In [33]:
from mnp import medallion, pairs
import pandas as pd
print('roles:', medallion.roles())
res = pairs.run(out_dir='reports/pairs')
t = res['table']
pd.set_option('display.width', 250, 'display.max_columns', 30)
cols = ['pair', 'error', 'stale', 'data_age_h', 'expected_from', 'used_of_expected',
        'p_low_in', 'p_low_in_random_walk', 'low_vs_random_walk', 'low_n_weeks',
        'p_high_in', 'p_high_in_random_walk', 'high_vs_random_walk', 'high_n_weeks']
print(t[[c for c in cols if c in t]].to_string(index=False))
for side in ('low_vs_random_walk', 'high_vs_random_walk'):
    if side in t: print(side, dict(t[side].value_counts()))

roles: {'calendar': 'calendar_events', 'carry': 'carry_features', 'candles': 'candles_1h', 'yields': 'world_bond_yields', 'positioning': 'cot_pair_sentiment', 'range_forecast': 'range_forecast_24h', 'regime': 'markov_regime_log'}
  pair  stale  data_age_h  expected_from  used_of_expected  p_low_in  p_low_in_random_walk    low_vs_random_walk  low_n_weeks  p_high_in  p_high_in_random_walk   high_vs_random_walk  high_n_weeks
EURGBP  False         1.8          ADR20          1.298315  0.993837              0.993343 same as a random walk          364   0.920833               0.928294 same as a random walk           196
EURAUD  False         2.8          ADR20          0.823748  0.980132              0.979624 same as a random walk          377   0.945148               0.911049   above a random walk           196
EURNZD  False         2.8          ADR20          0.623773  0.938202              0.944276 same as a random walk          269   0.887850               0.879074 same as a random walk 

In [34]:
from mnp import pairs
import pandas as pd
for when in ('2026-11-10 08:00', '2026-11-10 13:00'):
    t = pairs.run(out_dir='reports/pairs_replay', now=pd.Timestamp(when, tz='UTC'))['table']
    print('\n=====', when, 'UTC =====')
    print(t[['pair', 'hours_into_day', 'used_of_expected', 'p_low_in', 'p_low_in_random_walk', 'low_vs_random_walk',
             'p_high_in', 'p_high_in_random_walk', 'high_vs_random_walk']].to_string(index=False))
    for s in ('low_vs_random_walk', 'high_vs_random_walk'): print(s, dict(t[s].value_counts()))


===== 2026-11-10 08:00 UTC =====
  pair  hours_into_day  used_of_expected  p_low_in  p_low_in_random_walk    low_vs_random_walk  p_high_in  p_high_in_random_walk   high_vs_random_walk
EURGBP              10          0.442683  0.419355              0.374233 same as a random walk   0.162562               0.160757 same as a random walk
EURAUD              10          0.427943  0.488978              0.496623 same as a random walk   0.178667               0.179391 same as a random walk
EURNZD              10          0.439840  0.515306              0.497779 same as a random walk   0.536622               0.503955 same as a random walk
EURUSD              10          0.379763  0.304813              0.301961 same as a random walk   0.182469               0.150807   above a random walk
EURCAD              10          0.367869  0.268589              0.267097 same as a random walk   0.123162               0.117539 same as a random walk
EURCHF              10          0.338550  0.111765          

In [37]:
from mnp import pairs
import pandas as pd
pd.set_option('display.width', 250, 'display.max_columns', 30, 'display.max_colwidth', 90)
t = pairs.run(out_dir='reports/pairs')['table']
print(t[['pair', 'stale', 'data_age_h', 'p_low_in', 'p_low_in_random_walk', 'p_low_in_drifting_walk', 'low_tilt',
         'p_high_in', 'p_high_in_random_walk', 'p_high_in_drifting_walk', 'high_tilt']].to_string(index=False))
print(dict(t['low_tilt'].value_counts())); print(dict(t['high_tilt'].value_counts()))
print(t.loc[t['bars_note'].notna(), ['pair', 'bars_note']].to_string(index=False))

downloading medallion-serving/current/candles_1h_ig.parquet (0.7 MB) ... done in 1s
downloading medallion-serving/current/candles_1h_legfill.parquet (1.5 MB) ... 

done in 1s


  pair  stale  data_age_h  p_low_in  p_low_in_random_walk  p_low_in_drifting_walk                                               low_tilt  p_high_in  p_high_in_random_walk  p_high_in_drifting_walk                     high_tilt
EURGBP  False         2.0  0.993837              0.993343                0.992640                                                   none   0.920833               0.928294                 0.929506                          none
EURAUD  False         1.0  1.000000              1.000000                1.000000                                                   none   1.000000               1.000000                 1.000000                          none
EURNZD  False         1.0  1.000000              1.000000                1.000000                                                   none   1.000000               1.000000                 1.000000                          none
EURUSD  False         2.0  0.959677              0.947042                0.953254               

## 4. (Optional, later) fine-tune ModernBERT-large heads
Only after the Benzinga probe above confirms history depth and labels, and only when you have labelled headlines and want more accuracy. Saved heads in `heads/` are picked up automatically.
```
!python -m mnp.finetune driver --csv data/headlines.csv --out heads/driver
!python -m mnp.finetune tone   --csv data/headlines.csv --out heads/tone --max-length 2048
```